# Ingest circuits.csv file
### 1. Read the file using spark dataframe reader API
### 2. Add Metadata Columns 
-       Source File
-       Ingestion Timestamp
### 3. Write to bronze delta table

### Step 1 - Read the CSV file using the dataframe reader API

In [0]:
dbutils.widgets.text("p_batch_id", "")

In [0]:
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01_Environmnet_config

In [0]:
%run  ../00-common/02_bronze_helpers

In [0]:
source_file = f"{landing_folder_path}/{v_batch_id}/circuits.csv"
table_name = f"{catalog_name}.{bronze_schema}.circuits"

In [0]:
print(source_file)
print(table_name)

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DoubleType,
)

circuits_schema = StructType(
    [
        StructField("circuitId", StringType()),
        StructField("url", StringType()),
        StructField("circuitName", StringType()),
        StructField("lat", DoubleType()),
        StructField("long", DoubleType()),
        StructField("locality", StringType()),
        StructField("country", StringType()),
    ]
)

In [0]:
circuits_df = (
    spark.read.format("csv")
    .option("header", "true")
    .schema(circuits_schema)
    .load(source_file)
)

In [0]:
circuits_final_df = add_ingestion_medatat(circuits_df)

In [0]:
write_to_bronze(circuits_final_df, table_name, v_batch_id)

In [0]:
%sql
select * from formula1_incr.bronze.circuits
where batch_id = '2025-01'